# traffic-detection draft

AAI-521 · Team 2 · Starter notebook

How closely do automated counts of passing cars, overall and by color, agree with an independent tally? Run the setup and section definitions in order. Section 3 implements detection, tracking and counting. Data preparation, color classification and evaluation remain team tasks.

## 1. Shared Colab setup
Run these cells in order before starting the team sections.

### 1.1 Choose a GPU runtime

In Colab, choose Runtime → Change runtime type → GPU. Run this cell to see which GPU was assigned.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime, reconnect, and rerun this cell.")
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, round(gpu.total_memory / 2**30, 1), "GiB")

### 1.2 Install project packages

Start in a fresh Colab session. If Colab asks you to restart after installation, restart and continue with the setup cells.

In [ ]:
%pip install -q "transformers==4.57.3" accelerate "ultralytics>=8.3,<9" "lap>=0.5.12"

### 1.3 Log in to Hugging Face

Sign in on the [Cosmos model page](https://huggingface.co/nvidia/Cosmos-Reason2-2B) and complete any access prompt. Create a read token, save it in Colab Secrets as HF_TOKEN, and enable notebook access. If the secret is unavailable, the cell asks for the token privately.

In [ ]:
from getpass import getpass
from google.colab import userdata
from huggingface_hub import login

try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = getpass("Hugging Face read token: ")
try:
    login(token=token, add_to_git_credential=False)
finally:
    del token

### 1.4 Mount Drive and preview the input

Edit the path to point to the team clip. Use the preview to confirm the file and camera view.

In [ ]:
from pathlib import Path
from google.colab import drive
from PIL import Image
from IPython.display import display
import cv2

drive.mount("/content/drive")
video_path = Path("/content/drive/MyDrive/traffic/traffic.mp4")
if not video_path.is_file():
    raise FileNotFoundError(f"Edit video_path to point to your clip: {video_path}")
cap = cv2.VideoCapture(str(video_path))
try:
    ok, frame = cap.read()
finally:
    cap.release()
if not ok:
    raise ValueError("OpenCV could not decode the first frame.")
preview = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
preview.thumbnail((640, 640))
display(preview)

### 1.5 Load YOLO and check one frame

The first run downloads the detector weights. Zero detections on the preview is valid. Obinna implements tracking and line crossings in section 3.

In [ ]:
from ultralytics import YOLO

detector = YOLO("yolo11n.pt")
car_id = next(i for i, name in detector.names.items() if name == "car")
result = detector.predict(preview, classes=[car_id], verbose=False)[0]
print("Cars detected in the preview:", len(result.boxes))

### 1.6 Load Cosmos Reason 2 and its processor

This uses the model class, precision and Transformers release from the [first-timers guide](https://github.com/obiedeh/cosmos-reason2-first-timers-guide). The first run downloads the model weights.

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

model_id = "nvidia/Cosmos-Reason2-2B"
processor = AutoProcessor.from_pretrained(model_id)
cosmos = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id, dtype=torch.float16, device_map="auto", attn_implementation="sdpa"
).eval()

### 1.7 Check a Cosmos image call

Use a general question on the preview to confirm the processor and model work together. Tushar will choose the color labels and crop prompt in section 4.

In [ ]:
def ask_image(image, question):
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": question}
    ]}]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[prompt], images=[image], return_tensors="pt").to(cosmos.device)
    with torch.inference_mode():
        output = cosmos.generate(**inputs, max_new_tokens=48, do_sample=False)
    answer_tokens = output[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(answer_tokens, skip_special_tokens=True)[0].strip()

print(ask_image(preview, "Describe the scene in one short sentence."))

In [ ]:
# Use these column names when exchanging records between sections.
crossing_fields = ("track_id", "time_seconds", "crop_paths")
color_fields = ("track_id", "baseline_color", "cosmos_color")
manual_fields = ("time_seconds", "color")

## 2. Tushar Kummar | Prepare footage and define colors

Record the source, camera view and clip properties. Reserve a separate clip or time interval for evaluation before tuning. Choose color labels with the team, including an uncertain label for unreadable paint.

In [ ]:
def describe_clip(path):
    # TODO: Read frame size, frame rate and duration. Record source and split.
    raise NotImplementedError("Add clip metadata and split details.")

color_labels = []  # TODO: Enter the agreed labels before annotation.

**Observation and analysis:** Which camera or lighting conditions make paint color difficult to judge?

## 3. Obinna Edeh | Detect, track and count cars

Place crossings.py alongside this notebook in Colab or Databricks. Install ultralytics and lap using section 1.2. This section runs independently of the Cosmos setup. Set video_path to the local or /Volumes path of the team clip. Use full-resolution pixel coordinates for the counting line.

Each car is counted once per track ID when its bottom-center crosses the finite line in either direction. A three-pixel margin reduces jitter. ID changes can still cause missed or duplicate counts. Inspect the annotated video against the manual tally. Timestamps assume constant frame rate.


In [ ]:
from pathlib import Path
from crossings import track_crossings

# Set these after previewing the team's full-resolution video.
video_path = globals().get("video_path")
line_endpoints = None
output_dir = Path('outputs')

if video_path is None or line_endpoints is None:
    print('Set video_path and line_endpoints, then rerun this cell.')
else:
    crossings, annotated_path = track_crossings(
        video_path, line_endpoints, output_dir=output_dir
    )
    print('Counted cars:', len(crossings))
    print('Review video:', annotated_path)
    print('Share this entire run folder:', annotated_path.parent)


**Review and handoff**

The run folder contains annotated.mp4, crossings.csv, summary.json and crops/. A summary.json file marks a completed run. Each CSV row has track_id, time_seconds and crop_paths. Parse crop_paths with json.loads and resolve each crop relative to the run folder. Track IDs belong to one run and must be joined with colors within that run.

Juan can compare the crossing timestamps with the independent tally. Tushar can use up to three recent crops per crossing for color classification. Preserve the folder structure when sharing. Record actual missed cars, duplicate counts and ID changes after running the team clip. A readable but truncated input can stop decoding early, so also verify processed frames against the source duration.


## 4. Tushar Kummar | Compare color estimates

Use Obinna’s crops as the common input. Refer to sections 1.4 and 1.7 of the supplied Cosmos Reason 2 notebook for model setup and its `ask()` helper. Compare a simple color baseline and Cosmos on the same tracks.

In [ ]:
def label_track(crossing, labels):
    # TODO: Estimate a baseline color from the saved crops.
    # TODO: Use the shared ask_image() helper to request one label.
    # Return one record using color_fields. Keep the original track ID.
    raise NotImplementedError("Add baseline and Cosmos predictions.")

**Observation and analysis:** For a track where the methods disagree, inspect the crops and record what each method predicted.

## 5. Juan Norena | Evaluate and report

Record crossing times and colors independently before reviewing predictions. Match manual and predicted crossings one-to-one. Report the overall count, totals per color, missed cars and extra counts. Include uncertain predictions in the total.

In [ ]:
def evaluate_counts(crossings, color_records, manual_records):
    # TODO: Join predictions by track ID. Compare with manual_fields records.
    # TODO: Document the time tolerance and resolve ambiguous matches.
    # TODO: Summarize count errors and color accuracy on readable matched cars.
    raise NotImplementedError("Add count tables and error charts.")

**Observation and analysis:** Can the total match the manual tally even when individual events are wrong? Use a timestamped example from the held-out footage when results are available.